# Exploración inicial — datos fuente

Primer vistazo al *dataset* NYC TLC yellow taxi, restringido al **primer
mes disponible (enero de 2019)**, leyendo el fichero parquet original
directamente desde la fuente. La intención es hacerse una idea rápida
del contenido — antes de cualquier limpieza o ingesta — sin pasar por
la capa bronze ni por Spark.

Este notebook precede narrativamente a `preprocess.ipynb`: las
observaciones que se abren aquí se desarrollan allí sobre el conjunto
completo (ya en bronze) y se traducen en las reglas de limpieza que
aplica silver.

Objetivos:

- conocer las columnas del *feed* TLC y sus tipos,
- ver de un vistazo magnitudes y distribuciones a priori,
- detectar a alto nivel qué cosas requerirán limpieza más adelante.

In [ ]:
import os
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd

In [2]:
import matplotlib.pyplot as plt
import seaborn as sns

# Estilo académico: fondo blanco, rejilla sutil, tipografía neutra y paleta
# muted ("deep" de seaborn). Sustituye al estilo expresivo "fivethirtyeight"
# del ejemplo de referencia, demasiado informal para un informe.
sns.set_theme(
    style="whitegrid",
    context="paper",
    palette="deep",
    rc={
        "figure.figsize": (9, 5),
        "figure.dpi": 110,
        "savefig.dpi": 150,
        "font.size": 10,
        "axes.titlesize": 11,
        "axes.titleweight": "semibold",
        "axes.labelsize": 10,
        "axes.edgecolor": "#333333",
        "axes.linewidth": 0.8,
        "grid.color": "#dddddd",
        "grid.linewidth": 0.5,
        "xtick.labelsize": 9,
        "ytick.labelsize": 9,
        "lines.linewidth": 1.4,
        "patch.linewidth": 0.3,
        "legend.frameon": False,
        "legend.fontsize": 9,
    },
)
PALETTE = sns.color_palette("deep")

## 1. Subconjunto: enero de 2019

Trabajamos sobre el primer mes disponible (enero de 2019) y leemos
**directamente el fichero parquet original** publicado por NYC TLC, sin
pasar por la capa bronze. De este modo la exploración es independiente
del resto de la cadena (Spark, Nessie, Iceberg) y puede ejecutarse con
solo pandas. El fichero ronda los 100 MB y se descarga la primera vez
que se ejecuta el notebook.

La tabla de zonas (`taxi_zone_lookup.csv`, ~265 filas) se descarga
también de la fuente y se usa más abajo para etiquetar los conteos por
zona.

In [ ]:
PARQUET_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet"
ZONES_URL   = "https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv"

# El bind mount /opt/spark/src queda persistente entre ejecuciones del
# contenedor — se reusa el fichero descargado.
TMP_DIR = Path("/opt/spark/src/.tmp_explore")
TMP_DIR.mkdir(parents=True, exist_ok=True)
PARQUET_PATH = TMP_DIR / "yellow_tripdata_2019-01.parquet"
ZONES_PATH   = TMP_DIR / "taxi_zone_lookup.csv"

if not PARQUET_PATH.exists():
    print(f"Descargando {PARQUET_URL}")
    urlretrieve(PARQUET_URL, PARQUET_PATH)
    print(f"  -> {PARQUET_PATH.stat().st_size / 1024 / 1024:.1f} MB")
if not ZONES_PATH.exists():
    print(f"Descargando {ZONES_URL}")
    urlretrieve(ZONES_URL, ZONES_PATH)
    print(f"  -> {ZONES_PATH.stat().st_size / 1024:.1f} KB")

yt_jan19 = pd.read_parquet(PARQUET_PATH)
zl       = pd.read_csv(ZONES_PATH)

n_yt = len(yt_jan19)
n_zl = len(zl)
print(f"Yellow taxi (2019-01):  {n_yt:>10,} filas, {len(yt_jan19.columns)} columnas")
print(f"Taxi zone lookup:       {n_zl:>10,} filas, {len(zl.columns)} columnas")

## 2. Esquema y muestra

Empezamos viendo qué columnas trae el *feed* y de qué tipo, junto a unas
pocas filas reales para hacernos una idea del contenido.

In [ ]:
yt_jan19.dtypes

In [ ]:
yt_jan19.head()

## 3. Calidad de los datos: completitud

Antes de mirar las distribuciones conviene saber si hay columnas con
huecos significativos. Calculamos el **porcentaje de valores `NULL` por
columna** sobre los 7,7 M de registros — es la primera señal de qué
*features* podemos usar sin riesgos en pasos posteriores.

En el fichero original de enero de 2019 la columna `airport_fee` aparece
pero está íntegramente sin informar (se introduce con datos a partir de
2022) y `cbd_congestion_fee` ni siquiera existe todavía (se añade en
2025). El recargo `congestion_surcharge` está mayoritariamente a 0 —
todo es esperable, no patológico.

In [ ]:
null_pct = (yt_jan19.isna().mean() * 100).sort_values()

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(null_pct.index, null_pct.values,
        color=PALETTE[0], edgecolor="white", linewidth=0.4)
ax.set_xlabel("% de valores NULL")
ax.set_title(f"Completitud por columna  (n = {n_yt:,})")

xmax = max(null_pct.max(), 0.5)
ax.set_xlim(0, xmax * 1.18)
for i, pct in enumerate(null_pct.values):
    if pct > 0.05:
        ax.text(pct + xmax * 0.01, i, f"{pct:.2f}%",
                va="center", fontsize=9, color="#333333")
plt.tight_layout()
plt.show()

## 4. Distribución a priori de las variables numéricas

Resumen estadístico (recuento, media, desviación típica, mínimo, máximo
y percentiles representativos) de las principales variables continuas,
calculado sobre el **dataset completo**. Permite intuir la forma de cada
distribución: ¿simétrica?, ¿asimétrica con cola larga?, ¿hay valores
negativos imposibles? Las distancias entre p99 y `max` o entre `min` y
p1 anticipan dónde habrá que poner topes en silver.

In [ ]:
numeric_cols = [
    "trip_distance", "passenger_count", "fare_amount", "extra", "mta_tax",
    "tip_amount", "tolls_amount", "improvement_surcharge", "total_amount",
    "congestion_surcharge",
]
yt_jan19[numeric_cols].describe(
    percentiles=[0.01, 0.25, 0.5, 0.75, 0.95, 0.99]
).round(3)

A continuación los histogramas dan la forma visual.
**Recortamos a `[p1, p99]`** antes de pintar: el 1 % de outliers extremos
(p. ej. `fare_amount` con máximos > 600 000, o duraciones negativas por
desfases de reloj) colapsaría el histograma a una sola barra. La caja
entre p1 y p99 contiene el 98 % de las observaciones — basta para
apreciar asimetría, modas y colas.

Tanto los histogramas como el resto del análisis bivariado se calculan
sobre **todo el dataset** (~7,7 M de filas), no sobre una muestra:
trabajamos con el `DataFrame` completo en pandas, sin submuestreo.

In [ ]:
# Duración del viaje (minutos), calculada sobre el dataset completo.
# La reusaremos en histogramas, correlaciones y hexbin.
yt_jan19["trip_duration_min"] = (
    (yt_jan19["tpep_dropoff_datetime"] - yt_jan19["tpep_pickup_datetime"])
    .dt.total_seconds() / 60.0
)
print(f"Filas para los análisis (dataset completo): {len(yt_jan19):,}")

In [ ]:
hist_cols = [
    "trip_distance", "trip_duration_min", "fare_amount",
    "tip_amount", "total_amount",
]
fig, axes = plt.subplots(2, 3, figsize=(13.5, 7.5))
for ax, c in zip(axes.flat[:5], hist_cols):
    p1, p99 = yt_jan19[c].quantile([0.01, 0.99])
    ax.hist(yt_jan19[c].clip(p1, p99), bins=60,
            color=PALETTE[0], edgecolor="white", linewidth=0.3)
    ax.set_title(f"{c}\n[p1 = {p1:.2f}, p99 = {p99:.2f}]")
    ax.set_ylabel("frecuencia")

# passenger_count es discreto: bar plot por valor.
pc = yt_jan19["passenger_count"].dropna().astype(int).value_counts().sort_index()
ax = axes.flat[5]
ax.bar(pc.index.astype(str), pc.values,
       color=PALETTE[3], edgecolor="white", linewidth=0.3)
ax.set_title("passenger_count\n(variable discreta)")
ax.set_ylabel("frecuencia")

fig.suptitle(f"Distribuciones univariantes — dataset completo (n = {len(yt_jan19):,})",
             fontsize=12, fontweight="semibold")
fig.tight_layout()
plt.show()

## 5. Variables categóricas

Conteos por valor para los códigos que el *feed* utiliza: `VendorID`
(empresa que opera el taxímetro), `RatecodeID` (tipo de tarifa),
`payment_type` (forma de pago) y `store_and_fwd_flag` (registro local
diferido). Sirve para detectar valores que aparecen pero no figuran en el
diccionario TLC.

In [ ]:
# Bar chart por código categórico — escala log en y para que los valores
# minoritarios (RatecodeID=99, payment_type=0/4, NULLs) sean visibles junto
# a los dominantes. Anotamos el conteo exacto sobre cada barra.
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
specs = [
    ("VendorID", axes[0, 0]),
    ("RatecodeID", axes[0, 1]),
    ("payment_type", axes[1, 0]),
    ("store_and_fwd_flag", axes[1, 1]),
]
for col_name, ax in specs:
    counts = yt_jan19[col_name].astype(str).value_counts()
    bars = ax.bar(counts.index, counts.values,
                  color=PALETTE[0], edgecolor="white", linewidth=0.3)
    ax.set_yscale("log")
    ax.set_title(col_name)
    ax.set_ylabel("filas (escala log)")
    for bar, value in zip(bars, counts.values):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() * 1.08,
                f"{value:,}", ha="center", va="bottom",
                fontsize=8, color="#333333")

fig.suptitle("Conteos por código categórico (escala logarítmica)",
             fontsize=12, fontweight="semibold")
fig.tight_layout()
plt.show()

## 6. Patrón temporal

¿Cómo se distribuyen las recogidas a lo largo del mes y a lo largo del
día? Es la vista más básica para confirmar que la cobertura del mes es
completa y que aparece el patrón típico horario (valles de madrugada,
picos de mañana y tarde).

In [ ]:
day_df = yt_jan19["tpep_pickup_datetime"].dt.day.value_counts().sort_index()
hour_df = yt_jan19["tpep_pickup_datetime"].dt.hour.value_counts().sort_index()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
axes[0].plot(day_df.index, day_df.values, marker="o", color=PALETTE[0])
axes[0].fill_between(day_df.index, day_df.values, alpha=0.18, color=PALETTE[0])
axes[0].set_title("Recogidas por día del mes")
axes[0].set_xlabel("día del mes")
axes[0].set_ylabel("recogidas")
axes[0].set_xticks([1, 5, 10, 15, 20, 25, 31])

axes[1].plot(hour_df.index, hour_df.values, marker="o", color=PALETTE[3])
axes[1].fill_between(hour_df.index, hour_df.values, alpha=0.18, color=PALETTE[3])
axes[1].set_title("Recogidas por hora del día")
axes[1].set_xlabel("hora del día")
axes[1].set_ylabel("recogidas")
axes[1].set_xticks(range(0, 24, 2))

fig.tight_layout()
plt.show()

In [ ]:
# pandas dt.dayofweek: 0=Lunes ... 6=Domingo
heat = (
    pd.DataFrame({
        "dow":  yt_jan19["tpep_pickup_datetime"].dt.dayofweek,
        "hour": yt_jan19["tpep_pickup_datetime"].dt.hour,
    })
    .value_counts().reset_index(name="count")
)
labels = ["Lun", "Mar", "Mié", "Jue", "Vie", "Sáb", "Dom"]
heat["dow_label"] = heat["dow"].map(lambda i: labels[i])
pivot = (
    heat.pivot(index="dow_label", columns="hour", values="count")
    .reindex(labels)
)

fig, ax = plt.subplots(figsize=(13, 4.5))
sns.heatmap(
    pivot, cmap="Blues", linewidths=0.3, linecolor="white",
    cbar_kws={"label": "recogidas", "shrink": 0.8}, ax=ax,
)
ax.set_title("Recogidas por día de la semana × hora del día")
ax.set_xlabel("hora del día")
ax.set_ylabel("")
plt.tight_layout()
plt.show()

## 7. Zonas con más actividad

Cruzamos los conteos por `PULocationID` y `DOLocationID` con la tabla
`taxi_zone_lookup` para tener nombres legibles. El *top* da una idea
rápida de la geografía dominante de los viajes — esperaríamos Manhattan
y los aeropuertos (JFK, LGA, EWR) entre los primeros.

In [ ]:
# Cruzamos los conteos por PU/DO con la tabla de zonas para tener nombres
# legibles. Barras horizontales — los nombres son largos y el ranking se
# aprecia de un vistazo.
zone_lookup = zl.set_index("LocationID")[["Borough", "Zone"]]

top_pu = (
    yt_jan19["PULocationID"].value_counts().head(10)
    .rename_axis("LocationID").reset_index(name="count")
    .merge(zone_lookup, on="LocationID", how="left")
    .sort_values("count")
)
top_do = (
    yt_jan19["DOLocationID"].value_counts().head(10)
    .rename_axis("LocationID").reset_index(name="count")
    .merge(zone_lookup, on="LocationID", how="left")
    .sort_values("count")
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].barh(top_pu["Zone"], top_pu["count"],
             color=PALETTE[0], edgecolor="white", linewidth=0.3)
axes[0].set_title("Top 10 zonas de recogida")
axes[0].set_xlabel("recogidas")

axes[1].barh(top_do["Zone"], top_do["count"],
             color=PALETTE[3], edgecolor="white", linewidth=0.3)
axes[1].set_title("Top 10 zonas de destino")
axes[1].set_xlabel("recogidas")

fig.suptitle("Zonas con más actividad — enero 2019",
             fontsize=12, fontweight="semibold")
fig.tight_layout()
plt.show()

## 8. Relaciones entre variables

Una vez vistas las variables por separado, miramos cómo se relacionan
entre sí. Combinamos dos vistas:

- una **matriz de correlaciones** (Pearson) sobre las variables continuas,
  para detectar redundancias evidentes (`total_amount` ≈ suma de sus
  componentes), señales fuertes hacia variables de salida (`tip_amount`,
  `trip_duration_min`) y *features* casi constantes (poco poder
  discriminante);
- un **hexbin** del par con la correlación más alta —
  `fare_amount × trip_distance` — para ver la forma real de la relación
  más allá del coeficiente lineal: la banda principal corresponde a la
  tarifa por milla del taxímetro, y las bandas horizontales superiores
  delatan tarifas planas (p. ej. JFK ↔ Manhattan, ~52 USD).

Ambas vistas se calculan sobre **todo el dataset** (~7,7 M de filas).
Las columnas íntegramente nulas en este mes (p. ej. `airport_fee`) se
excluyen automáticamente para que no aparezcan filas vacías en la
matriz.

In [ ]:
corr_cols = [
    "trip_distance", "trip_duration_min", "passenger_count",
    "fare_amount", "extra", "mta_tax", "tip_amount", "tolls_amount",
    "improvement_surcharge", "congestion_surcharge", "total_amount",
]
# Filtramos columnas íntegramente nulas o ausentes en este mes — producen
# filas vacías (toda NaN) que distorsionan la lectura de la matriz.
present_cols = [
    c for c in corr_cols
    if c in yt_jan19.columns and yt_jan19[c].notna().any()
]
dropped = [c for c in corr_cols if c not in present_cols]
if dropped:
    print(f"Columnas excluidas (vacías o ausentes): {dropped}")

# Recortamos a [p1, p99] antes de correlacionar: los outliers extremos
# (fares de 6 cifras, duraciones negativas por desfases de reloj) distorsionan
# Pearson y harían parecer débiles relaciones que en realidad son fuertes.
clip_lo = yt_jan19[present_cols].quantile(0.01)
clip_hi = yt_jan19[present_cols].quantile(0.99)
corr = (
    yt_jan19[present_cols]
    .clip(lower=clip_lo, upper=clip_hi, axis=1)
    .corr()
)

# Mostramos solo el triángulo inferior — la matriz es simétrica.
mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
fig, ax = plt.subplots(figsize=(11, 8))
sns.heatmap(
    corr, mask=mask, annot=True, fmt=".2f",
    cmap="RdBu_r", center=0, vmin=-1, vmax=1, square=True,
    linewidths=0.4, linecolor="white",
    annot_kws={"size": 9},
    cbar_kws={"label": "correlación de Pearson", "shrink": 0.7},
    ax=ax,
)
ax.set_title("Matriz de correlaciones — variables numéricas\n(dataset completo, recorte p1–p99)")
plt.tight_layout()
plt.show()

In [ ]:
# Hexbin del par más fuerte: fare_amount × trip_distance, sobre el dataset
# completo. Hexbin agrega densidad y escala bien con millones de puntos.
clip_d = yt_jan19["trip_distance"].quantile([0.01, 0.99])
clip_f = yt_jan19["fare_amount"].quantile([0.01, 0.99])
mask = (
    yt_jan19["trip_distance"].between(clip_d.iloc[0], clip_d.iloc[1])
    & yt_jan19["fare_amount"].between(clip_f.iloc[0], clip_f.iloc[1])
)
sub = yt_jan19.loc[mask, ["trip_distance", "fare_amount"]]

fig, ax = plt.subplots(figsize=(9, 6))
hb = ax.hexbin(
    sub["trip_distance"], sub["fare_amount"],
    gridsize=45, cmap="Blues", mincnt=1, bins="log",
)
fig.colorbar(hb, ax=ax, label="log10(viajes por hex)", shrink=0.85)
ax.set_title("Densidad bivariada: trip_distance × fare_amount\n(dataset completo, recorte p1–p99)")
ax.set_xlabel("trip_distance (millas)")
ax.set_ylabel("fare_amount (USD)")
plt.tight_layout()
plt.show()

## 9. Observaciones de esta primera mirada

Lo que esta exploración sugiere y que se profundiza en `preprocess.ipynb`
con el conjunto completo:

- **Completitud**: la mayoría de columnas están casi siempre informadas.
  En el fichero de enero de 2019, `airport_fee` aparece pero está
  íntegramente sin informar y `cbd_congestion_fee` ni siquiera existe
  todavía como columna — ambos campos se introducen en años posteriores,
  esperable y no patológico.
- **Magnitudes sospechosas** en importes y distancia: mínimos negativos y
  máximos muy alejados del p99 (623 K USD en `fare_amount` o
  `total_amount`). Conviene investigar tanto la cola inferior como la
  superior.
- **Códigos categóricos**: aparecen valores fuera del diccionario TLC
  (`VendorID=4/5`, `RatecodeID=99`, `payment_type=0`) que la escala log
  permite ver junto a los dominantes; hay que verificar formalmente
  cuáles son válidos.
- **Timestamps**: la duración del viaje calculada desde los *timestamps*
  presenta valores negativos y extremos (de minutos a > 24 h) — refleja
  desfases de reloj que la limpieza tendrá que cribar.
- **Esquema**: el *feed* mezcla tipos numéricos heterogéneos entre años;
  bronze los normaliza tras un *schema drift* histórico.
- **Volumen**: un solo mes ya ronda los millones de filas — pandas lo
  maneja en memoria, pero sobre el histórico completo se justifica el
  paso a Spark/Iceberg.
- **Patrón temporal**: días laborables presentan dos picos (mañana y
  tarde-noche), mientras que los fines de semana tienen el pico desplazado
  a la madrugada — útil para *features* horario/día como entrada de modelo.
- **Relaciones**: la matriz de correlaciones confirma la redundancia de
  `total_amount` con sus componentes y el vínculo fuerte
  `trip_distance ↔ fare_amount ↔ trip_duration_min`. Variables casi
  constantes (`mta_tax`, `improvement_surcharge`) aportan poca señal.
  El *hexbin* `fare_amount × trip_distance` muestra la banda lineal
  dominante (tarifa por milla del taxímetro) y bandas horizontales que
  delatan **tarifas planas** — la más visible alrededor de 52 USD
  corresponde a la tarifa fija JFK ↔ Manhattan.

A partir de aquí, `preprocess.ipynb` recorre los Q1 completos (2019, 2020,
2022, 2023 y 2025) y formaliza, con evidencia cuantitativa, las reglas de
limpieza que aplica `src/etl/silver/clean_yellow_taxi.py`.